## Read Me

This is a flexible notebook for stepping through an annotation step by step given a specific scenario input. The notebook must be modified in order to select the directory and input file. This is not ideal, so we will have to come up with a better system!

## Set up

In [1]:
import json # !pip
import os
import textwrap
import pandas as pd
import importlib
import numpy as np
from pathlib import Path
import sys

/Users/anna/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/anna/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
ROOT_DIR = os.getcwd() + '/../'
sys.path.append(ROOT_DIR)

sys.path.append(ROOT_DIR+'src/')
print(ROOT_DIR)

/Users/anna/Documents/code/graph_extract/run_annotation/../


In [3]:
import src.annotate_scenario as annotate_scenario
import src.prompts as prompts
import src.translate_to_vis as translate_to_vis
import src.node as node
import src.get_emb_distances as get_emb_distances
import src.utils as utils
import src.moral_projection as moral_projection

import src.core_process as core_process
importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)
importlib.reload(prompts)
importlib.reload(node)
importlib.reload(utils)
importlib.reload(core_process)

<module 'src.core_process' from '/Users/anna/Documents/code/graph_extract/run_annotation/../src/core_process.py'>

In [4]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

## Selet Scenario Input File

In [5]:
# set main paths
SCENARIO_DIR = ROOT_DIR + "scenarios_inputs/" + "cheung_variants/"
# DATA_DIR_HUMAN = ROOT_DIR + "human_data/" 
OUTPUT_DIR = ROOT_DIR + "annotated_outputs/" + "cheung_variants/"

In [6]:
#set scenario file filename
FILENAME = 'robinhood.json'

#select scenario and action choice
SCENARIO_ID = 1
ACT_ID = '1'

#read in the scenario
scenario_json = utils.open_scenario(SCENARIO_DIR, FILENAME, SCENARIO_ID, ACT_ID)


I am an employee of a bank in Germany. I have many loyal customers who trust me and my bank. Among
them is one family of five who is deeply in debt and does not fulfill the conditions for being
granted another loan. I see an undetectable way to falsify the family's credit score, allowing them
to qualify for the loan. At the same time, I would abuse the trust of the other customers and of my
colleagues. But if I do this, I will be able to prevent the family from going bankrupt. 




In [ ]:
# print scenario json
# print(json.dumps(scenario_json, indent=4))

### Pretest deontology 

In [21]:
core_process.process_values_minimal("get an abortion")

-68.0

In [22]:
core_process.process_values_minimal("get a vasectomy")

-33.0

In [23]:
core_process.process_values_minimal("use contraception")

11.0

#### There are 4 major stages of processing.

0. Entities
Label the entities (no human data)

1. Value Scores / "Deontology"
Score the action in moral value

2. Outcomes
Map action to probable outcomes 

3. Outcome Utilities
Consequentialist analysis of harms/benefits of each outcome to each entity

4. Outcome Links
Connection between each entity and each outcome, in terms of Cause, Intend, and Desire



## Go through annotation process step by step (replicates main function in annotate_scenario) 

In [17]:

# get the action choice and convert to two pronoun options (I and Ziv)
this_act = scenario_json['options'][ACT_ID]
# this_act_I = this_act
# this_act_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_act_I)

# print(this_act_Ziv)
# print(this_act_I)

#get the scenario and convert to two pronoun options
this_scenario = scenario_json['text']
print(scenario_json['text'])
this_scenario_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_scenario)
print("\n\nScenario:")
print(this_scenario_Ziv)
# print(this_scenario)


# create a dictionary to write out to csv later
scenario_dict = {'scenario': this_scenario, 'scenario_idx': scenario_json['id'],
                    'choice': this_act}


print('\n\nAction choice:') 
print(this_act)


I am an employee of a bank in Germany. I have many loyal customers who trust me and my bank. Among them is one family of five who is deeply in debt and does not fulfill the conditions for being granted another loan. I see an undetectable way to falsify the family's credit score, allowing them to qualify for the loan. At the same time, I would abuse the trust of the other customers and of my colleagues. But if I do this, I will be able to prevent the family from going bankrupt.


Scenario:
Ziv is an employee of a bank in Germany. They have many loyal customers who trust them and their bank. Among them is one family of five who is deeply in debt and does not fulfill the conditions for being granted another loan. Ziv sees an undetectable way to falsify the family's credit score, allowing them to qualify for the loan. At the same time, they would abuse the trust of the other customers and of their colleagues. But if they do this, they will be able to prevent the family from going bankrupt.

In [18]:

#initialize Graph object    
g = annotate_scenario.node.Graph()
g.reset()   
print('Graph g initialized and reset.')
g.set_version('d6d780129095bb9540a7957befb2014ea42d92c1')



Graph g initialized and reset.


#### Step 0. Get entities

In [19]:
#Step 0. Get entities

# identify all sentient beings, returning both pronoun forms and a string list
returned_beings = core_process.process_beings(this_scenario,this_act,g)
beings_I = returned_beings[0]
beings_Ziv = returned_beings[1]
beings_str_list = returned_beings[2]
g = returned_beings[3]
    
#update the scenario dict with the beings
scenario_dict["entities"] = beings_str_list



Identified these entities: 

I
family of 5 customers in debt
other loyal customers
colleagues


#### Step 1. Deontology / Action Value Scores

In [20]:
#Step 1.  #ACTION VALUE SCORES

#call the process_values function to rate the moral goodness or wrongness of the action with no context
deontic_value,g  =  core_process.process_values_simple(this_act,g) 
print(deontic_value)

    

-71.0


#### Step 2. Anticipated Outcomes

In [26]:
#Step 2. Outcomes
processed_events = core_process.process_outcomes(this_scenario, this_act)
events_I = processed_events[1]
events_Ziv = processed_events[0]
print("\n".join(events_Ziv))         
scenario_dict["outcomes"]= events_I

[autoreload of embedding_utils failed: Traceback (most recent call last):
  File "/Users/anna/anaconda3/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 273, in check
    superreload(m, reload, self.old_objects)
  File "/Users/anna/anaconda3/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 471, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/Users/anna/anaconda3/lib/python3.11/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 940, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/Users/anna/Documents/code/graph_extract/run_annotation/../src/embedding_utils.py", line 11, in <module>
    OPENAI_API_KEY = utils.resolve_openai_api_key()
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: module 'src.utils' has no 

The family receives the loan funds
The family avoids immediate bankruptcy
The bank takes on a loan to a borrower who does not meet its lending standards
Ziv deceives the bank
Ziv betrays the trust of Ziv's colleagues
Ziv abuses the trust of Ziv's other customers


#### Step 3. Outcome Utilities

In [27]:
events_I

['The family receives the loan funds',
 'The family avoids immediate bankruptcy',
 'The bank takes on a loan to a borrower who does not meet its lending standards',
 'I deceive the bank',
 'I betray the trust of my colleagues',
 'I abuse the trust of my other customers']

In [29]:
core_process.process_util_minimal(events_I[0])

23.0

In [30]:
core_process.process_util_minimal(events_I[1])

43.0

In [31]:
core_process.process_util_minimal(events_I[2])

-19.0

In [ ]:
core_process.process_util_minimal(events_I[4])

-43.0

In [34]:
core_process.process_util_minimal(events_I[5])

-44.0

In [ ]:
#Step 3. Outcome utilities

impacts_list = core_process.process_impacts(this_scenario_Ziv, this_act, events_Ziv, events_I,beings_Ziv,g) 


#### Step 4. Cause / Intend / Know Links

In [ ]:
#Step 4. causal / intentional / knowledge links -- run on currently generated event/outcome list
output_links = annotate_scenario.process_causal_links(this_scenario_Ziv, events_Ziv, events_I, this_act,g)    

#### Step 5. Write out the results

In [ ]:
narrative = FILENAME.split('.')[0]
this_output_filename = f"{OUTPUT_DIR}/{narrative}_{SCENARIO_ID}_choice_{ACT_ID}.json"

In [ ]:
#optional -- write out the results 
print('\n\nWriting to file: '+this_output_filename)
g_print = g.print_graph()
utils.write_jsonlines(this_output_filename, g_print)
print('\n\n')


translate_to_vis.main(this_output_filename)
